# MedGemma Experimentation Notebook

This notebook runs MedGemma on a fixed set of 20 medical images (10 melanoma + 10 nevus) to test prompt variations.

## Instructions

1. Run cells 1-4 to set up the environment (only needed once per session)
2. Edit the CONFIGURATION cell (prompt and settings)
3. Run the experiment cell
4. View results and accuracy

**Important: Change only ONE thing at a time** between experiments to understand what affects results.

---

## Cell 1: Install Dependencies

In [ ]:
!pip install -q transformers>=4.45.0 accelerate>=0.26.0 datasets>=3.0.0 pillow>=10.0.0 torch>=2.0.0
print("[OK] Dependencies installed")

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_PATH = "/content/drive/MyDrive/MedGemma_Results"
os.makedirs(RESULTS_PATH, exist_ok=True)
print(f"[OK] Results will save to: {RESULTS_PATH}")

## Cell 3: Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"[OK] GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[ERROR] No GPU. Go to Runtime > Change runtime type > T4 GPU")

## Cell 4: Load Model and Dataset

First run downloads about 15GB. Takes 5-7 minutes.

In [ ]:
import os
import json
import re
from datetime import datetime
from transformers import AutoProcessor, AutoModelForImageTextToText
from datasets import load_dataset, concatenate_datasets
import torch

# ---------- HUGGING FACE TOKEN ----------
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Replace with your token
# ----------------------------------------

os.environ["HF_TOKEN"] = HF_TOKEN

print("Loading MedGemma...")
MODEL_ID = "google/medgemma-4b-it"
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, token=HF_TOKEN, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
print("[OK] Model loaded")

print("Loading dataset...")
ds_raw = load_dataset("MartiHan/Open-MELON-VL-2.5K")
dataset = concatenate_datasets([ds_raw["train"], ds_raw["validation"], ds_raw["test"]])
print(f"[OK] Dataset loaded: {len(dataset)} images")

# Fixed test set: first 10 melanoma + first 10 nevus from indices
MELANOMA_INDICES = [3, 4, 5, 6, 7, 8, 9, 12, 20, 22]
NEVUS_INDICES = [33, 43, 118, 119, 120, 121, 170, 171, 196, 197]
print(f"[OK] Test set: {len(MELANOMA_INDICES)} melanoma + {len(NEVUS_INDICES)} nevus")

---

## CONFIGURATION

Edit the prompt and settings below, then run the experiment cell.

### Example Prompts

```python
# Baseline pathologist prompt
PROMPT = "You are a pathologist examining a biopsy. Describe the key histological features in 2-3 sentences. Provide your best guess for the diagnosis and your certainty level (high/low). Answer format: Description. Diagnosis. Confidence."

# Binary choice (constrains output)
PROMPT = "You are a pathologist examining a biopsy. This lesion is either MELANOMA or a BENIGN NEVUS. Describe the key histological features in 2-3 sentences. State your diagnosis (melanoma or benign nevus) and certainty (high/low)."

# Simple description
PROMPT = "Describe what you see in this medical image in 2-3 sentences."

# Question format
PROMPT = "What type of lesion is shown in this image? Is it benign or malignant? Explain your reasoning."

# Differential diagnosis
PROMPT = "As a dermatopathologist, provide your top differential diagnosis for this lesion. Focus on whether this appears benign or malignant."
```

### Parameter Guidelines

- **temperature**: 0.0 = deterministic, 0.7 = balanced, 1.5+ = creative/random
- **top_k**: 1 = greedy, 50 = balanced, 100 = diverse
- **top_p**: 0.1 = focused, 0.95 = balanced, 1.0 = all tokens
- **max_tokens**: 100-300 for short responses, 500+ for detailed. Usually it its too long it will start generating usless things as next steps or sth else.
- **do_sample**: False = ignore temperature/top_k/top_p and just pick the most likely token, True = use the parameters above

In [ ]:
# ============================================================
# CONFIGURATION - Edit these values, then run the next cell
# ============================================================

PROMPT = """You are a pathologist examining a biopsy. This lesion is either MELANOMA or a BENIGN NEVUS. Describe the key histological features in 2-3 sentences. State your diagnosis (melanoma or benign nevus) and certainty (high/low)."""

SETTINGS = {
    "temperature": 0.0,    # 0.0 = deterministic, higher = more random
    "top_k": 50,           # Number of top tokens to consider
    "top_p": 0.95,         # Cumulative probability threshold
    "max_tokens": 200,     # Maximum response length
    "do_sample": True,    # ignore temperature/top_k/top_p and just pick the most likely token, True = use the parameters above
}

# ============================================================
print("[OK] Configuration set. Run the next cell to start the experiment.")

---

## Run Experiment

Processes all 20 images and shows results.

In [ ]:
import time

def generate(image, prompt, settings):
    """Generate caption for a single image."""
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=settings["max_tokens"],
            do_sample=settings["do_sample"],
            temperature=settings["temperature"] if settings["do_sample"] else None,
            top_k=settings["top_k"] if settings["do_sample"] else None,
            top_p=settings["top_p"] if settings["do_sample"] else None,
        )
    
    response = processor.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant response
    if "model" in response.lower():
        response = response.split("model")[-1].strip()
    return response

def classify_response(text):
    """Extract prediction from response using keyword matching."""
    text_lower = text.lower()
    has_melanoma = "melanoma" in text_lower
    has_nevus = any(kw in text_lower for kw in ["nevus", "nevi", "naevus", "naevi", "benign"])
    
    if has_melanoma and not has_nevus:
        return "melanoma"
    elif has_nevus and not has_melanoma:
        return "nevus"
    elif has_melanoma and has_nevus:
        return "melanoma"  # melanoma takes priority in ambiguous cases
    else:
        return "unknown"

# Run experiment
print("="*70)
print("EXPERIMENT")
print("="*70)
print(f"Prompt: {PROMPT[:80]}...")
print(f"Settings: {SETTINGS}")
print("="*70)

results = []
all_indices = [(idx, "melanoma") for idx in MELANOMA_INDICES] + [(idx, "nevus") for idx in NEVUS_INDICES]

for i, (idx, ground_truth) in enumerate(all_indices):
    print(f"\n[{i+1}/{len(all_indices)}] Image {idx} (ground truth: {ground_truth})")
    print("-"*70)
    
    sample = dataset[idx]
    gt_caption = sample.get("caption", "N/A")
    
    print(f"Ground Truth Caption: {gt_caption[:150]}..." if len(gt_caption) > 150 else f"Ground Truth Caption: {gt_caption}")
    
    start = time.time()
    response = generate(sample["image"], PROMPT, SETTINGS)
    elapsed = time.time() - start
    
    prediction = classify_response(response)
    correct = prediction == ground_truth
    status = "CORRECT" if correct else "WRONG"
    
    print(f"Generated Response: {response}")
    print(f"Prediction: {prediction} | {status} ({elapsed:.1f}s)")
    
    results.append({
        "index": idx,
        "ground_truth": ground_truth,
        "ground_truth_caption": gt_caption,
        "response": response,
        "prediction": prediction,
        "correct": correct,
        "time_seconds": round(elapsed, 2)
    })

# Calculate accuracy
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)

total = len(results)
correct = sum(1 for r in results if r["correct"])
melanoma_results = [r for r in results if r["ground_truth"] == "melanoma"]
nevus_results = [r for r in results if r["ground_truth"] == "nevus"]
unknown_count = sum(1 for r in results if r["prediction"] == "unknown")

melanoma_correct = sum(1 for r in melanoma_results if r["correct"])
nevus_correct = sum(1 for r in nevus_results if r["correct"])

accuracy = correct / total * 100
sensitivity = melanoma_correct / len(melanoma_results) * 100 if melanoma_results else 0
specificity = nevus_correct / len(nevus_results) * 100 if nevus_results else 0
balanced_accuracy = (sensitivity + specificity) / 2

print(f"Overall Accuracy:    {correct}/{total} = {accuracy:.1f}%")
print(f"Sensitivity (melanoma): {melanoma_correct}/{len(melanoma_results)} = {sensitivity:.1f}%")
print(f"Specificity (nevus):    {nevus_correct}/{len(nevus_results)} = {specificity:.1f}%")
print(f"Balanced Accuracy:   {balanced_accuracy:.1f}%")
print(f"Unknown predictions: {unknown_count}")

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_data = {
    "timestamp": datetime.now().isoformat(),
    "prompt": PROMPT,
    "settings": SETTINGS,
    "metrics": {
        "accuracy": round(accuracy, 2),
        "sensitivity": round(sensitivity, 2),
        "specificity": round(specificity, 2),
        "balanced_accuracy": round(balanced_accuracy, 2),
        "unknown_count": unknown_count
    },
    "results": results
}

filepath = f"{RESULTS_PATH}/experiment_{timestamp}.json"
with open(filepath, "w") as f:
    json.dump(save_data, f, indent=2)

print(f"\n[SAVED] {filepath}")

---

## View All Saved Experiments

In [ ]:
import os
import json

print("Saved Experiments")
print("="*70)

files = sorted([f for f in os.listdir(RESULTS_PATH) if f.endswith(".json")])
for f in files:
    path = os.path.join(RESULTS_PATH, f)
    with open(path) as fp:
        data = json.load(fp)
    m = data.get("metrics", {})
    print(f"\n{f}")
    print(f"  Prompt: {data.get('prompt', '')[:60]}...")
    print(f"  Accuracy: {m.get('accuracy', 0):.1f}% | Balanced: {m.get('balanced_accuracy', 0):.1f}%")
    print(f"  Sensitivity: {m.get('sensitivity', 0):.1f}% | Specificity: {m.get('specificity', 0):.1f}%")

---

## Troubleshooting

**No GPU found**: Go to Runtime > Change runtime type > T4 GPU

**Out of memory**: Reduce max_tokens or restart runtime (Runtime > Restart runtime)

**Token error**: Check your HF_TOKEN is correct and you accepted MedGemma license on HuggingFace